# 1.0 Import libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

# 2.0 Data load

In [3]:
data = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_10_30_icarda_data_formating_for_dbvisualizer/icarda_data.csv')
grasses_categories = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_10_30_icarda_data_formating_for_dbvisualizer/grasses_categories.csv')
legumes_categories = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_10_30_icarda_data_formating_for_dbvisualizer/legumes_categories.csv')

In [4]:
print(data.shape)
data.head(2)

(6604, 16)


,id_lab,functional_group,tax_name,id,genus,run,replication,dm,ndf,om,gas_g_dm_incubated,ch4_g_dm_incubated,ch4_ndf,ch4_g_digested,tddm,ch4_g_kg_dm
0,250100028.0,Legume,Medicago sativa,101045,Medicago,1,1.0,92.765482,92.76548217,88.03689603,201.4431243,28.90708834,28.90708834,40.75785051,70.92397655,16.04393415
1,250100028.0,Legume,Medicago sativa,101045,Medicago,1,2.0,92.765482,39.23080084,88.03689603,179.8429447,24.41046826,5.772119889,34.38596823,70.98962023,13.54823221


In [5]:
print(grasses_categories.shape)
grasses_categories.head(2)

(212, 14)


,CH4_Decrease_Pct,Benchmark_vs_Control,CH4_Category,TDDM_Category,Z_Score_CH4,Z_Score_TDDM,Composite_Score,Overall_Rank,CH4_8h_Reduction_Pct,CH4_24h_Reduction_Pct,Accession_ID,CH4_Intensity,TDDM,lmf_category
0,74.473265,41.291073,Low,High,0.886946,1.349323,3.123215,1,NaN,NaN,250200460,24.954109,90.705977,Category_1a
1,75.522652,43.704557,Low,High,0.922053,1.176595,3.020701,2,NaN,NaN,250200468,23.928263,86.945646,Category_1a


In [6]:
print(legumes_categories.shape)
legumes_categories.head(2)

(250, 14)


,CH4_Decrease_Pct,Benchmark_vs_Control,CH4_Category,TDDM_Category,Z_Score_CH4,Z_Score_TDDM,Composite_Score,Overall_Rank,CH4_8h_Reduction_Pct,CH4_24h_Reduction_Pct,Accession_ID,CH4_Intensity,TDDM,lmf_category
0,39.425434,20.813715,Low,High,2.662375,0.523053,5.847802,1,NaN,NaN,250100193,32.813719,75.994373,Category_1a
1,36.395142,16.852356,Low,High,2.350180,0.987399,5.687759,2,NaN,NaN,250200367,34.455252,79.697061,Category_1a


# 3.0 Average estimation

In [7]:
data.columns

Index(['id_lab', 'functional_group', 'tax_name', 'id', 'genus', 'run',
       'replication', 'dm', 'ndf', 'om', 'gas_g_dm_incubated',
       'ch4_g_dm_incubated', 'ch4_ndf', 'ch4_g_digested', 'tddm',
       'ch4_g_kg_dm'],
      dtype='object')

In [8]:
#Columns processing for nutrition complete dataframe
category_columns = ['id_lab', 'functional_group', 'tax_name', 'id', 'genus', 'run', 'replication']
numeric_columns = ['dm', 'ndf', 'om', 'gas_g_dm_incubated', 'ch4_g_dm_incubated', 'ch4_ndf', 'ch4_g_digested', 'tddm', 'ch4_g_kg_dm']

for df in [data]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [9]:
# Group by subset and id, keeping categorical columns for complete dataframe
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # # Count the number of replicates in each group
    # mean_df['n_replicates_nutrition'] = (
    #     df.groupby([ 'id_lab'])
    #       .size()
    # )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [10]:
# Creation of averaged dataframe
data_average = mean_with_replicates(data, category_columns, numeric_columns)

/tmp/ipykernel_40658/3206661856.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_40658/3206661856.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]


In [11]:
print(data_average.shape)
data_average.head(2)

(760, 16)


,id_lab,functional_group,tax_name,id,genus,run,replication,dm,ndf,om,gas_g_dm_incubated,ch4_g_dm_incubated,ch4_ndf,ch4_g_digested,tddm,ch4_g_kg_dm
0,250100028.0,Legume,Medicago sativa,101045,Medicago,1,1.0,92.77,46.88,88.04,191.27,27.20,9.58,41.44,65.73,15.09
1,250100029.0,Legume,Medicago sativa,101063,Medicago,1,1.0,94.98,50.67,89.58,194.48,29.19,5.47,54.50,53.87,16.20


# 4.0 Formating

## 4.1 Averaged dataframe

In [12]:
data_average.functional_group.unique()

['Legume', 'Grass', 'CC', 'CH', 'C5', 'C6', 'grass']
Categories (7, object): ['C5', 'C6', 'CC', 'CH', 'Grass', 'Legume', 'grass']

In [13]:
data_average["functional_group"] = data_average["functional_group"].replace({"Grass": "Grasses", 'grass': 'Grasses', 'Legume': 'Herbaceous_legumes'})

/tmp/ipykernel_40658/2579225967.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  data_average["functional_group"] = data_average["functional_group"].replace({"Grass": "Grasses", 'grass': 'Grasses', 'Legume': 'Herbaceous_legumes'})


In [14]:
data_average.functional_group.unique()

['Herbaceous_legumes', 'Grasses', 'CC', 'CH', 'C5', 'C6']
Categories (6, object): ['C5', 'C6', 'CC', 'CH', 'Grasses', 'Herbaceous_legumes']

In [15]:
data_average = data_average.rename(columns={'ch4_g_digested': 'methane_intensity'})

In [16]:
data_average.columns

Index(['id_lab', 'functional_group', 'tax_name', 'id', 'genus', 'run',
       'replication', 'dm', 'ndf', 'om', 'gas_g_dm_incubated',
       'ch4_g_dm_incubated', 'ch4_ndf', 'methane_intensity', 'tddm',
       'ch4_g_kg_dm'],
      dtype='object')

In [17]:
data_average_2 = data_average[['id_lab', 'id', 'tax_name', 'functional_group', 'genus', 'dm', 'ndf', 'om', 'gas_g_dm_incubated',
       'ch4_g_dm_incubated', 'ch4_ndf', 'ch4_g_kg_dm', 'methane_intensity', 'tddm']]

In [18]:
print(data_average_2.shape)
data_average_2.head(2)

(760, 14)


,id_lab,id,tax_name,functional_group,genus,dm,ndf,om,gas_g_dm_incubated,ch4_g_dm_incubated,ch4_ndf,ch4_g_kg_dm,methane_intensity,tddm
0,250100028.0,101045,Medicago sativa,Herbaceous_legumes,Medicago,92.77,46.88,88.04,191.27,27.20,9.58,15.09,41.44,65.73
1,250100029.0,101063,Medicago sativa,Herbaceous_legumes,Medicago,94.98,50.67,89.58,194.48,29.19,5.47,16.20,54.50,53.87


# 4.2 Categories dataframes

In [19]:
grasses_categories.columns

Index(['CH4_Decrease_Pct', 'Benchmark_vs_Control', 'CH4_Category',
       'TDDM_Category', 'Z_Score_CH4', 'Z_Score_TDDM', 'Composite_Score',
       'Overall_Rank', 'CH4_8h_Reduction_Pct', 'CH4_24h_Reduction_Pct',
       'Accession_ID', 'CH4_Intensity', 'TDDM', 'lmf_category'],
      dtype='object')

In [25]:
grasses_target_columns = grasses_categories[['Accession_ID', 'CH4_Category', 'TDDM_Category', 'lmf_category', 'Overall_Rank']]
grasses_target_columns = grasses_target_columns.rename(columns={'Accession_ID': 'id_lab', 'Overall_Rank': 'lmf_category_rank', 'CH4_Category': 'ch4_category', 'TDDM_Category': 'tddm_category'})
print(grasses_target_columns.shape)
grasses_target_columns.head(2)

(212, 5)


,id_lab,ch4_category,tddm_category,lmf_category,lmf_category_rank
0,250200460,Low,High,Category_1a,1
1,250200468,Low,High,Category_1a,2


In [26]:
legumes_categories.columns

Index(['CH4_Decrease_Pct', 'Benchmark_vs_Control', 'CH4_Category',
       'TDDM_Category', 'Z_Score_CH4', 'Z_Score_TDDM', 'Composite_Score',
       'Overall_Rank', 'CH4_8h_Reduction_Pct', 'CH4_24h_Reduction_Pct',
       'Accession_ID', 'CH4_Intensity', 'TDDM', 'lmf_category'],
      dtype='object')

In [27]:
legumes_target_columns = legumes_categories[['Accession_ID', 'CH4_Category', 'TDDM_Category', 'lmf_category', 'Overall_Rank']]
legumes_target_columns = legumes_target_columns.rename(columns={'Accession_ID': 'id_lab', 'Overall_Rank': 'lmf_category_rank', 'CH4_Category': 'ch4_category', 'TDDM_Category': 'tddm_category'})
print(legumes_target_columns.shape)
legumes_target_columns.head(2)

(250, 5)


,id_lab,ch4_category,tddm_category,lmf_category,lmf_category_rank
0,250100193,Low,High,Category_1a,1
1,250200367,Low,High,Category_1a,2


# 5.0 Categories concat and merge with averaged data

## 5.1 Categories concat

In [28]:
lmf_categories = pd.concat([grasses_target_columns, legumes_target_columns])
print(lmf_categories.shape)
lmf_categories.head(2)

(462, 5)


,id_lab,ch4_category,tddm_category,lmf_category,lmf_category_rank
0,250200460,Low,High,Category_1a,1
1,250200468,Low,High,Category_1a,2


## 5.2 Merge of averaged data with categories

In [29]:
icarda_compiled = pd.merge(data_average_2, lmf_categories, on="id_lab", how="outer")
print(icarda_compiled.shape)
icarda_compiled.head(2)

(760, 18)


,id_lab,id,tax_name,functional_group,genus,dm,ndf,om,gas_g_dm_incubated,ch4_g_dm_incubated,ch4_ndf,ch4_g_kg_dm,methane_intensity,tddm,ch4_category,tddm_category,lmf_category,lmf_category_rank
0,250100028.0,101045,Medicago sativa,Herbaceous_legumes,Medicago,92.77,46.88,88.04,191.27,27.20,9.58,15.09,41.44,65.73,Medium,High,Category_1b,54.0
1,250100029.0,101063,Medicago sativa,Herbaceous_legumes,Medicago,94.98,50.67,89.58,194.48,29.19,5.47,16.20,54.50,53.87,High,Not High,Category_4,14.0


In [31]:
icarda_compiled.duplicated().sum()

np.int64(0)

In [32]:
icarda_compiled["subset"] = 1
icarda_compiled["subset"] = icarda_compiled["subset"].astype("int64")

cols = ["subset"] + [col for col in icarda_compiled.columns if col != "subset"]
icarda_compiled = icarda_compiled[cols]

In [33]:
icarda_compiled.head(2)

,subset,id_lab,id,tax_name,functional_group,genus,dm,ndf,om,gas_g_dm_incubated,ch4_g_dm_incubated,ch4_ndf,ch4_g_kg_dm,methane_intensity,tddm,ch4_category,tddm_category,lmf_category,lmf_category_rank
0,1,250100028.0,101045,Medicago sativa,Herbaceous_legumes,Medicago,92.77,46.88,88.04,191.27,27.20,9.58,15.09,41.44,65.73,Medium,High,Category_1b,54.0
1,1,250100029.0,101063,Medicago sativa,Herbaceous_legumes,Medicago,94.98,50.67,89.58,194.48,29.19,5.47,16.20,54.50,53.87,High,Not High,Category_4,14.0


# 6.0 Save

In [34]:
icarda_compiled.to_csv('/content/drive/MyDrive/lmf/output/2026_10_30_icarda_data_formating_for_dbvisualizer/icarda_compiled.csv', index=False)